# Telescope Runs — Stage 2 Demo (classical run ingest)

This notebook demonstrates `solsys_code/management/commands/load_telescope_runs.py`
(issue #37 Stage 2), the classical-schedule ingest command that parses run lines
and idempotently creates one `CalendarEvent` per observing night.

It demonstrates:

- Seeding `Observatory` records for the 4 supported sites (idempotent `update_or_create`)
- Writing a small sample schedule file to a temporary path
- Invoking `load_telescope_runs` via `call_command`
- Inspecting the resulting `CalendarEvent` rows (title, start/end times, description)
- Re-running the command to confirm idempotency (no duplicate events)

This notebook lives in `pre_executed/` because it is **DB-dependent** (it seeds
`Observatory` records and creates `CalendarEvent` rows) and is therefore **NOT
run during Sphinx/CI/ReadTheDocs builds**, per `docs/notebooks/README.md`.

## Django setup

Standard boilerplate to make `src.fomo.settings` importable from this notebook's
location (`docs/notebooks/pre_executed/` — three levels under the repo root, so
`parents[2]` gives the repo root) and to allow synchronous ORM calls inside
Jupyter's async event loop.

In [1]:
import os
import sys
from pathlib import Path

import django

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
# NOTE: parents[2] is correct only when the Jupyter kernel CWD is
# docs/notebooks/pre_executed/. Start Jupyter from that directory, or
# adjust the index if you launch from the repo root.
repo_root_path = Path.cwd().resolve().parents[2]
assert (
    repo_root_path / 'manage.py'
).exists(), (
    f'Repo root not found at {repo_root_path}. Run Jupyter from docs/notebooks/pre_executed/ or adjust parents[] index.'
)
repo_root = str(repo_root_path)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

django.setup()

## Seed Observatory records

`load_telescope_runs` resolves telescope names to `Observatory` rows via MPC obscode
(through `telescope_runs.get_site()`). The 4 sites below match the values used in
`solsys_code/tests/test_telescope_runs.py`'s `setUpTestData`. Using
`update_or_create` makes this cell idempotent — safe to re-run against any dev DB.

In [2]:
from solsys_code.solsys_code_observatory.models import Observatory

SITE_DATA = {
    '268': dict(
        name='Magellan Clay Telescope',
        short_name='Magellan-Clay',
        lat=-29.0146,
        lon=-70.6926,
        altitude=2402,
        timezone='America/Santiago',
    ),
    '269': dict(
        name='Magellan Baade Telescope',
        short_name='Magellan-Baade',
        lat=-29.0146,
        lon=-70.6926,
        altitude=2402,
        timezone='America/Santiago',
    ),
    '809': dict(
        name='ESO, La Silla',
        short_name='NTT',
        lat=-29.2567,
        lon=-70.7300,
        altitude=2347,
        timezone='America/Santiago',
    ),
    'E10': dict(
        name='Siding Spring Observatory',
        short_name='FTS',
        lat=-31.2734,
        lon=149.0612,
        altitude=1149,
        timezone='Australia/Sydney',
    ),
}

for obscode, fields in SITE_DATA.items():
    obs, created = Observatory.objects.update_or_create(obscode=obscode, defaults=fields)
    action = 'created' if created else 'updated/unchanged'
    print(f'  obscode={obscode!r:>4}  {action:>18}  short_name={obs.short_name!r}')

  obscode='268'   updated/unchanged  short_name='Magellan-Clay'
  obscode='269'   updated/unchanged  short_name='Magellan-Baade'
  obscode='809'   updated/unchanged  short_name='NTT'
  obscode='E10'   updated/unchanged  short_name='FTS'


## Night convention: ESO noon-to-noon vs Las Campanas both-inclusive

Different sites count the nights of a date range differently, and
`_iter_run_nights` applies the right convention per site:

- **Las Campanas (Magellan)** — Start and End are *both* inclusive
  observing nights, so a range yields `E - S + 1` nights (see
  `docs/design/telescope_runs_calendar.rst` "Night convention").
- **ESO sites (`ESO_NOON_TO_NOON_SITES`, e.g. NTT / La Silla)** — the range
  is transcribed verbatim from ESO's *Tatoo* tool, whose displayed **End date
  is the noon-to-noon closing boundary of the last night**, not itself an
  observing night. So the last observing night is `End - 1` and the range
  yields `E - S` nights.

For example, ESO's Tatoo reports *4.0 nights* for `NTT ... 9-13 July`; the
cell below confirms `_iter_run_nights` produces exactly those 4 nights
(9–12 July), while the same-length Magellan range stays both-inclusive.

In [3]:
from solsys_code.management.commands.load_telescope_runs import _iter_run_nights
from solsys_code.telescope_runs import ESO_NOON_TO_NOON_SITES, parse_run_line

print('ESO noon-to-noon sites:', sorted(ESO_NOON_TO_NOON_SITES))
print()

for line in ['NTT EFOSC2 allocation 9-13 July', 'Magellan-Baade IMACS 17-18 July']:
    parsed = parse_run_line(line)
    nights = _iter_run_nights(parsed)
    convention = (
        'ESO Tatoo noon-to-noon (drops End boundary -> E - S nights)'
        if parsed.telescope in ESO_NOON_TO_NOON_SITES
        else 'Las Campanas both-inclusive (E - S + 1 nights)'
    )
    print(f'{line!r}')
    print(f'  telescope={parsed.telescope}  day1={parsed.day1} day2={parsed.day2}  ->  {len(nights)} nights')
    print(f'  convention: {convention}')
    print(f'  nights: {[d.isoformat() for d in nights]}')
    print()

ESO noon-to-noon sites: ['NTT']

'NTT EFOSC2 allocation 9-13 July'
  telescope=NTT  day1=9 day2=13  ->  4 nights
  convention: ESO Tatoo noon-to-noon (drops End boundary -> E - S nights)
  nights: ['2026-07-09', '2026-07-10', '2026-07-11', '2026-07-12']

'Magellan-Baade IMACS 17-18 July'
  telescope=Magellan-Baade  day1=17 day2=18  ->  2 nights
  convention: Las Campanas both-inclusive (E - S + 1 nights)
  nights: ['2026-07-17', '2026-07-18']



## Write a sample schedule file

A real schedule file is a plain-text file with one run line per non-blank line.
Here we write a small sample to a temporary file. The format accepted by
`parse_run_line` is flexible: the month name may appear before or after the
day range, and instrument names may be hyphenated.

In [4]:
import tempfile

SAMPLE_SCHEDULE = """\
NTT EFOSC2 allocation 9-13 July
FTS MUSCAT4 allocation 10-12 July
Magellan IMACS 14-16 July (proposed)
"""

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as tmp:
    tmp.write(SAMPLE_SCHEDULE)
    schedule_path = tmp.name

print('Schedule file written to:', schedule_path)
print()
print('Contents:')
print(SAMPLE_SCHEDULE)

Schedule file written to: /tmp/tmpcn50w9m7.txt

Contents:
NTT EFOSC2 allocation 9-13 July
FTS MUSCAT4 allocation 10-12 July
Magellan IMACS 14-16 July (proposed)



## Invoke the load_telescope_runs command

`call_command` is the Django-recommended way to invoke management commands
programmatically. The command will:

1. Parse each run line with `parse_run_line`
2. Resolve the telescope name to its `Observatory` via `get_site`
3. Expand the date range to one evening per night
4. For each night: compute `sun_event` sunset/sunrise and the -15° dark window
5. Create or update a `CalendarEvent` keyed on `(telescope, instrument, start_time)`

The end-of-run summary reports `created`, `updated`, `unchanged`, and `skipped` counts.

In [5]:
import io

from django.core.management import call_command

stdout_buf = io.StringIO()
stderr_buf = io.StringIO()

call_command('load_telescope_runs', schedule_path, stdout=stdout_buf, stderr=stderr_buf)

print('stdout:', stdout_buf.getvalue())
if stderr_buf.getvalue():
    print('stderr (skipped lines):', stderr_buf.getvalue())

stdout: Done. lines processed: 3, created: 0, updated: 7, unchanged: 0, skipped: 1

stderr (skipped lines): Line 3: Ambiguous telescope 'Magellan': matches multiple SITES keys ['Magellan-Clay', 'Magellan-Baade']; use a more specific telescope name (e.g. "Magellan-Clay" or "Magellan-Baade"). (line text: 'Magellan IMACS 14-16 July (proposed)')



## Inspect the created CalendarEvent rows

Each observing night in the schedule produces one `CalendarEvent`. The `description`
field (D-06) contains the -15° dark-window UTC times, the run status, and the
original source line for traceability.

In [6]:
from tom_calendar.models import CalendarEvent

events = CalendarEvent.objects.order_by('start_time')
print(f'Total CalendarEvent rows: {events.count()}')
print()

for ev in events:
    print(f'Title      : {ev.title}')
    print(f'start_time : {ev.start_time.isoformat()}')
    print(f'end_time   : {ev.end_time.isoformat()}')
    print('Description:')
    for line in ev.description.splitlines():
        print(f'  {line}')
    print()

Total CalendarEvent rows: 26

Title      : 3I/ATLAS (demo): DCT
start_time : 2025-07-05T02:50:31+00:00
end_time   : 2025-07-05T12:10:16+00:00
Description:

Title      : 3I/ATLAS (demo): DCT
start_time : 2025-07-06T02:50:20+00:00
end_time   : 2025-07-06T12:10:48+00:00
Description:

Title      : Crash Test Campaign: Sinistro
start_time : 2026-05-11T15:56:04+00:00
end_time   : 2026-05-12T05:10:31+00:00
Description:

Title      : Test Campaign: 0.5m + CMOS
start_time : 2026-05-25T18:49:30+00:00
end_time   : 2026-05-26T03:36:46+00:00
Description:

Title      : Test Campaign: NOT/ALFOSC
start_time : 2026-05-25T20:08:45+00:00
end_time   : 2026-05-26T06:08:06+00:00
Description:

Title      : FTS MuSCAT
start_time : 2026-07-08T08:45:00+00:00
end_time   : 2026-07-08T15:35:00+00:00
Description:
  Queue observation — LCO2026A-003
  Fixed field, 08:45–15:35 UTC

Title      : FTS MuSCAT
start_time : 2026-07-09T08:45:00+00:00
end_time   : 2026-07-09T15:35:00+00:00
Description:
  Queue observation — L

## Cancelled classical run: [CANCELLED] title prefix (D-01/D-02)

A staff member marks a classical run cancelled by adding the recognized `cancelled`
status word/parenthetical to the source schedule line and re-running
`load_telescope_runs`. The resulting `CalendarEvent.title` now begins with
`[CANCELLED] `, mirroring the LCO sync's existing `[CANCELLED]`/`[EXPIRED]` title-prefix
idiom (see `sync_lco_observation_calendar.py`'s `_FAILURE_PREFIX_BY_STATUS`). No
templatetag change was needed for this: `[CANCELLED]` is already a member of
`calendar_display_extras._TERMINAL_PREFIXES`, so the event also picks up the terminal
box-shadow ring for free.

Below we reuse the NTT/EFOSC2 telescope+instrument already loaded above (so its
`Observatory` record already exists) but on a new night, with the `cancelled`
status word appended.

In [7]:
CANCELLED_SCHEDULE = """\
NTT EFOSC2 21-22 July (cancelled)
"""

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as tmp:
    tmp.write(CANCELLED_SCHEDULE)
    cancelled_path = tmp.name

stdout_buf_c = io.StringIO()
call_command('load_telescope_runs', cancelled_path, stdout=stdout_buf_c, stderr=io.StringIO())
print(stdout_buf_c.getvalue())

cancelled_event = CalendarEvent.objects.get(
    telescope='NTT', instrument='EFOSC2', start_time__date__day=21, start_time__date__month=7
)
print(f'Title      : {cancelled_event.title}')
print('Description:')
for line in cancelled_event.description.splitlines():
    print(f'  {line}')
assert cancelled_event.title.startswith('[CANCELLED] '), 'Expected a [CANCELLED]-prefixed title'

Done. lines processed: 1, created: 0, updated: 0, unchanged: 1, skipped: 0

Title      : [CANCELLED] NTT EFOSC2
Description:
  Dark window (-15 deg, UTC): 2026-07-21T23:14:24+00:00 to 2026-07-22T10:24:12+00:00
  Status: cancelled
  Source line: NTT EFOSC2 21-22 July (cancelled)


## Idempotency check

Running the command a second time with the same file should produce zero new
events and zero updates — the summary should report `created: 0, updated: 0`.
This satisfies INGEST-03.

In [8]:
stdout_buf2 = io.StringIO()
stderr_buf2 = io.StringIO()

call_command('load_telescope_runs', schedule_path, stdout=stdout_buf2, stderr=stderr_buf2)

print('Second run stdout:', stdout_buf2.getvalue())
count_after = CalendarEvent.objects.count()
print(f'CalendarEvent count unchanged: {count_after}')

Second run stdout: Done. lines processed: 3, created: 0, updated: 0, unchanged: 7, skipped: 1

CalendarEvent count unchanged: 26


## Cross-session drift tolerance (idempotency-key robustness)

The event `start_time` is a *computed* sun-event time (`telescope_runs.sun_event()`),
not a stable external identifier. Between independent ingests of the same night days or
weeks apart, astropy refreshes its IERS Earth-orientation data (UT1-UTC / polar motion),
so the recomputed sunset can drift by a second or two. `load_telescope_runs` therefore
matches an existing event whose `start_time` is within a few minutes of the freshly
computed value (a proximity window, not an exact datetime), so a drifted re-ingest
**updates** the existing night instead of silently creating a near-duplicate row.

Below we simulate that cross-session drift by shifting the recomputed sun-event times by
+2 seconds on a third ingest of the same schedule file. The row count stays constant and
the summary reports `created: 0` — no duplicates. (The computed `end_time` and the
dark-window times embedded in `description` drift by the same +2s, so full-night events
report as `updated` rather than `unchanged`; the key point is that none are re-`created`.)

In [9]:
from unittest import mock

import astropy.units as u

from solsys_code import telescope_runs as tr

_real_sun_event = tr.sun_event


def _sun_event_plus_2s(site, d, kind):
    """Real sun_event() with every crossing shifted +2s (mimics cross-session IERS drift)."""
    setting, rising = _real_sun_event(site, d, kind)
    return setting + 2 * u.s, rising + 2 * u.s


count_before_drift = CalendarEvent.objects.count()
stdout_buf3 = io.StringIO()
with mock.patch(
    'solsys_code.management.commands.load_telescope_runs.sun_event',
    side_effect=_sun_event_plus_2s,
):
    call_command('load_telescope_runs', schedule_path, stdout=stdout_buf3, stderr=io.StringIO())

print('Drifted (+2s) re-ingest stdout:', stdout_buf3.getvalue())
print(f'CalendarEvent count before drift: {count_before_drift}, after: {CalendarEvent.objects.count()}')
print('No duplicates were created (created: 0); the drifted nights were matched and updated.')

Drifted (+2s) re-ingest stdout: Done. lines processed: 3, created: 0, updated: 7, unchanged: 0, skipped: 1

CalendarEvent count before drift: 26, after: 26
No duplicates were created (created: 0); the drifted nights were matched and updated.


## Partial-night window tokens

Run lines may carry a trailing `(BoN|HHMM)-(EoN|HHMM)` token to restrict the
event to a portion of the night.  `load_telescope_runs` passes the parsed
`start_window` / `end_window` through `_resolve_window_time`, which converts:

- `BoN` → computed sunset for that night
- `EoN` → computed sunrise for that night
- `HHMM` → a fixed UTC datetime (HHMM < 1200 → next-morning UTC; ≥ 1200 → same evening)

This is used for e.g. shared-telescope runs where only the first or second half
of each night is allocated.

In [10]:
import tempfile, io
from django.core.management import call_command
from tom_calendar.models import CalendarEvent

PARTIAL_SCHEDULE = """\
Magellan-Clay Lightspeed 18-20 July BoN-0626
"""

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as tmp:
    tmp.write(PARTIAL_SCHEDULE)
    partial_path = tmp.name

# Clear any existing Clay events so counts are clean for this demo
CalendarEvent.objects.filter(telescope='Magellan-Clay').delete()

stdout_buf = io.StringIO()
call_command('load_telescope_runs', partial_path, stdout=stdout_buf, stderr=io.StringIO())
print(stdout_buf.getvalue())

# Inspect: end_time should be 06:26 UTC on d+1 (not computed sunrise ~11:26 UTC)
print(f'{"Night":10}  {"start_time (UTC)":22}  {"end_time (UTC)":22}  {"duration (h)":>12}')
print('-' * 72)
for ev in CalendarEvent.objects.filter(telescope='Magellan-Clay').order_by('start_time'):
    dur = (ev.end_time - ev.start_time).total_seconds() / 3600
    print(
        f'{str(ev.start_time.date()):10}  {ev.start_time.strftime("%Y-%m-%d %H:%M:%S"):22}'
        f'  {ev.end_time.strftime("%Y-%m-%d %H:%M:%S"):22}  {dur:>11.2f}h'
    )

Done. lines processed: 1, created: 3, updated: 0, unchanged: 0, skipped: 0

Night       start_time (UTC)        end_time (UTC)          duration (h)
------------------------------------------------------------------------
2026-07-18  2026-07-18 22:11:29     2026-07-19 06:26:00            8.24h
2026-07-19  2026-07-19 22:12:01     2026-07-20 06:26:00            8.23h
2026-07-20  2026-07-20 22:12:32     2026-07-21 06:26:00            8.22h


## Summary

This notebook demonstrates the Phase 03 `load_telescope_runs` command satisfying
requirements INGEST-01, INGEST-02, and INGEST-03:

| Requirement | Description | Demonstrated by |
|-------------|-------------|------------------|
| INGEST-01 | One `CalendarEvent` per observing night, `start_time=sunset`, `end_time=sunrise` | 4 NTT (ESO noon-to-noon: 9–12 July) + 3 FTS events created; Magellan line skipped (ambiguous telescope name) |
| INGEST-02 | `title`, `description` (dark window, status, source line), `telescope`, `instrument` populated | `ev.description` printed above shows all three D-06 lines |
| INGEST-03 | Idempotent re-run leaves row count and field values unchanged | Second run reports `created: 0, updated: 0` |
| INGEST-03 (drift) | Re-ingest whose computed `start_time` drifted (IERS refresh) matches within a tolerance window instead of duplicating | +2s-drifted third run reports `created: 0`; row count unchanged |

The command is invoked via `call_command` — equivalent to running
`./manage.py load_telescope_runs <filepath>` from the shell.

This notebook is **pre-executed** and intentionally excluded from automated doc
builds (not referenced in `docs/notebooks.rst`) because it depends on
`Observatory` records and live `CalendarEvent` writes in the local dev DB.